# 06 — Learning to rank

**What this shows**

- A learned ranker (LambdaMART) is compared with the strongest *single* hand-built signal on Microsoft's MSLR-WEB10K, the standard learning-to-rank benchmark.
- The single feature is chosen on the validation split, never on test, so the baseline is as strong as one feature can honestly be.
- The comparison uses the same ruler as notebooks 01 to 05: `ndcg_at_k` from `ragexp.metrics` and the paired bootstrap over test queries.

## The prediction, written before running

> **LambdaMART beats the best single feature.**
>
> One feature sees one signal. A listwise tree ensemble combines all 136 and optimises
> the ranking metric directly. On LTR's home benchmark that should show.
>
> **Falsified if** the paired-bootstrap 95% CI on nDCG@10 (LambdaMART minus best single
> feature, Fold 1 test queries) includes 0 or lies below it.

This refines the row first written in the README ("LambdaMART wins"), which named no
baseline. The refinement was committed on 2026-10-29, before any model was trained or any
feature was scored. The README row also asked whether a published null was due to its
LLM-generated labels. MSLR-WEB10K ships only numeric features and human grades, with no
query or document text, so nothing here can be relabelled. That question stays open as a
caveat.

## Setup, fixed in advance

- **Data:** MSLR-WEB10K, Fold 1 only: `train.txt`, `vali.txt`, `test.txt`. 136 features,
  relevance grades 0 to 4. Read with `sklearn.datasets.load_svmlight_file(query_id=True)`
  from a local copy outside the repo (the licence does not allow redistribution).
- **LambdaMART:** `xgboost.XGBRanker(objective="rank:ndcg", n_estimators=2000,
  learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
  tree_method="hist", eval_metric="ndcg@10", early_stopping_rounds=100, random_state=0)`.
  Early stopping watches the validation split. No other tuning.
- **Best single feature:** for each of the 136 features, in both directions (higher is
  better, lower is better), rank each validation query's documents by that value and score
  mean nDCG@10. The best of those 272 rankings is the baseline. Ties keep file order.
- **Metric:** nDCG@10 on the test split with `ragexp.metrics.ndcg_at_k`. A query with no
  document graded above 0 is undefined (`None`) and dropped, as in notebook 01.
- **Test:** `paired_bootstrap` with the default seed, queries in file order.

In [1]:
import os
import time
from pathlib import Path

import numpy as np
import xgboost

from ragexp.ltr import best_single_feature, grades_by_query, group_index, load_split, ndcg_by_query, query_groups, rank_by_scores
from ragexp.metrics import paired_bootstrap
from ragexp.runs import metric_vector, save_scores

FOLD = Path(os.environ.get("MSLR_FOLD", Path.home() / "data/mslr-web10k/extracted/Fold1"))
split = {}
for name in ("train", "vali", "test"):
    X, y, qid = load_split(FOLD / f"{name}.txt")
    split[name] = (X, y, qid, query_groups(qid))
    print(f"{name:>5}: {X.shape[0]:>7} rows, {len(split[name][3]):>5} queries, {X.shape[1]} features, "
          f"grades {dict(zip(*np.unique(y, return_counts=True)))}")
print("xgboost", xgboost.__version__)

train:  723412 rows,  6000 queries, 136 features, grades {np.int64(0): np.int64(377957), np.int64(1): np.int64(232569), np.int64(2): np.int64(95082), np.int64(3): np.int64(12658), np.int64(4): np.int64(5146)}


 vali:  235259 rows,  2000 queries, 136 features, grades {np.int64(0): np.int64(121522), np.int64(1): np.int64(75815), np.int64(2): np.int64(31910), np.int64(3): np.int64(4209), np.int64(4): np.int64(1803)}


 test:  241521 rows,  2000 queries, 136 features, grades {np.int64(0): np.int64(124784), np.int64(1): np.int64(77896), np.int64(2): np.int64(32459), np.int64(3): np.int64(4450), np.int64(4): np.int64(1932)}
xgboost 3.3.0


## The baseline: the best single feature

Every one of the 136 features, in both directions, ranks the validation queries. The
best of the 272 rankings becomes the baseline. Test is not looked at.

In [2]:
Xv, yv, qv, gv = split["vali"]
t0 = time.time()
col, sign, vali_ndcg = best_single_feature(Xv, yv, gv)
print(f"best single feature: column {col} (feature id {col + 1}), "
      f"{'higher' if sign > 0 else 'lower'} is better, vali nDCG@10 = {vali_ndcg:.4f}  ({time.time() - t0:.0f}s)")

best single feature: column 133 (feature id 134), higher is better, vali nDCG@10 = 0.3379  (29s)


## LambdaMART

The settings are the pre-registered ones. Early stopping watches validation nDCG@10.

In [3]:
Xt, yt, qt, gt = split["train"]
ranker = xgboost.XGBRanker(
    objective="rank:ndcg", n_estimators=2000, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, tree_method="hist", eval_metric="ndcg@10",
    early_stopping_rounds=100, random_state=0,
)
t0 = time.time()
ranker.fit(Xt, yt, qid=group_index(gt), eval_set=[(Xv, yv)], eval_qid=[group_index(gv)], verbose=False)
print(f"trained in {time.time() - t0:.0f}s; best iteration {ranker.best_iteration}, "
      f"vali nDCG@10 (xgboost's own gain) {ranker.best_score:.4f}")

trained in 68s; best iteration 796, vali nDCG@10 (xgboost's own gain) 0.5125


## Scoring on test

Both systems rank the same test queries and are scored with the repo's `ndcg_at_k`.

In [4]:
Xs, ys, qs, gs = split["test"]
grades = grades_by_query(ys, gs)
runs = {
    "ltr_best_feature": rank_by_scores(sign * Xs[:, col], gs),
    "ltr_lambdamart": rank_by_scores(ranker.predict(Xs), gs),
}
scores = {name: {q: {"ndcg@10": v} for q, v in ndcg_by_query(r, grades).items()} for name, r in runs.items()}
for name, s in scores.items():
    save_scores(name, s)

order = [q for q, _, _ in gs]
a = metric_vector(scores["ltr_best_feature"], "ndcg@10", order)
b = metric_vector(scores["ltr_lambdamart"], "ndcg@10", order)
n_def = sum(x is not None for x in a)
print(f"test queries: {len(order)}, defined (some doc graded > 0): {n_def}")
for name, v in (("best single feature", a), ("LambdaMART", b)):
    print(f"{name:>20}: mean nDCG@10 {np.mean([x for x in v if x is not None]):.4f}")
delta, (lo, hi), p = paired_bootstrap(a, b)
print(f"LambdaMART - best feature: delta {delta:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]  p={p:.4f}")

test queries: 2000, defined (some doc graded > 0): 1928
 best single feature: mean nDCG@10 0.3272
          LambdaMART: mean nDCG@10 0.4922
LambdaMART - best feature: delta +0.1651  CI [+0.1552, +0.1748]  p=0.0000


## Resolving the prediction

The prediction was **"LambdaMART beats the best single feature"**. It held, by a wide margin.

| | test nDCG@10 | Δ vs best feature (paired) |
|---|---|---|
| best single feature (feature 134, picked on vali) | 0.3272 | baseline |
| LambdaMART (`rank:ndcg`, stopped at iteration 796) | **0.4922** | +0.1651, CI [+0.1552, +0.1748], p ≈ 0 |

The mean is over the 1,928 of 2,000 test queries with at least one document graded above 0.
The other 72 are undefined and dropped, as in notebook 01.

1. **The margin is large.** The CI's lower edge is +0.155, so the prediction would have held
   even against a far stronger baseline than the falsifier required. The best feature scored
   0.3379 on vali and 0.3272 on test, so picking it on vali did not flatter it much.
2. **This is the expected direction on this benchmark, not a surprise.** MSLR's 136
   features were built for learned rankers to combine. The notebook confirms the instrument
   works on a learning-to-rank task. It does not compare against published MSLR numbers:
   those use other gain conventions and all five folds.
3. **Not tested here: labels.** The published null that motivated the first README row used
   LLM-generated labels. MSLR has no query or document text, so nothing here can be
   relabelled, and that question stays open.

Scope: one fold, one fixed hyperparameter setting, no tuning on test. The model's own
vali score (0.5125) uses xgboost's nDCG gain, not `ragexp.metrics`, so it is not comparable
with the table above.